In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/Internship project/05_processed_text_dataset.csv"
)

print(df.shape)

df.head()

(1206, 11)


,Message_ID,Person,Chat_Name,Timestamp,Sender,Message,Risk_Label,Risk_Category,Confidence,Processed_Message,Tokens
0,1,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:15:00,Vishnu,"Da, report kandille?",Normal,NaN,High,da report kandille,"['da', 'report', 'kandille']"
1,2,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:17:00,You,Kandu. Ellam okay alle?,Normal,NaN,High,kandu ellam okay alle,"['kandu', 'ellam', 'okay', 'alle']"
2,3,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:18:00,Vishnu,Mostly okay. Pakshe aa last item kurachu stran...,Suspicious,Coded Language,Medium,mostly okay pakshe aa last item kurachu strang...,"['mostly', 'okay', 'pakshe', 'aa', 'last', 'it..."
3,4,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:20:00,You,Entha issue?,Normal,NaN,High,entha issue,"['entha', 'issue']"
4,5,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:21:00,Vishnu,Numbers match cheyyunnilla. Randu places il di...,Suspicious,Coded Language,Medium,number match cheyyunnilla randu place il diffe...,"['number', 'match', 'cheyyunnilla', 'randu', '..."


In [ ]:
X = df["Processed_Message"].fillna("")

y = df["Risk_Label"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

#Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr.fit(
    X_train_tfidf,
    y_train
)

lr_pred = lr.predict(X_test_tfidf)

print("===== Logistic Regression =====")

print(classification_report(
    y_test,
    lr_pred
))

===== Logistic Regression =====
              precision    recall  f1-score   support

   High Risk       0.28      0.36      0.31        14
      Normal       0.84      0.81      0.82       182
  Suspicious       0.34      0.35      0.34        46

    accuracy                           0.70       242
   macro avg       0.48      0.51      0.49       242
weighted avg       0.71      0.70      0.70       242



#Linear SVM

In [ ]:
svm = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm.fit(
    X_train_tfidf,
    y_train
)

svm_pred = svm.predict(
    X_test_tfidf
)

print("===== Linear SVM =====")

print(classification_report(
    y_test,
    svm_pred
))

===== Linear SVM =====
              precision    recall  f1-score   support

   High Risk       0.31      0.36      0.33        14
      Normal       0.83      0.87      0.85       182
  Suspicious       0.44      0.35      0.39        46

    accuracy                           0.74       242
   macro avg       0.53      0.52      0.52       242
weighted avg       0.73      0.74      0.73       242



#Cross Validation

In [ ]:
scores = cross_val_score(
    svm,
    vectorizer.transform(X),
    y,
    cv=5,
    scoring="f1_macro"
)

print(scores)

print("Average F1:", scores.mean())

[0.35338006 0.38055413 0.38233362 0.37878972 0.44332863]
Average F1: 0.38767723186529046


#Hyperparameter Tuning

In [ ]:
param_grid = {
    "C":[0.1,1,10],
    "loss":["hinge","squared_hinge"]
}

grid = GridSearchCV(
    LinearSVC(class_weight="balanced"),
    param_grid,
    cv=5,
    scoring="f1_macro"
)

grid.fit(
    X_train_tfidf,
    y_train
)

print(grid.best_params_)

print(grid.best_score_)

{'C': 1, 'loss': 'squared_hinge'}
0.40842642264394013


#Best Model

In [ ]:
best_model = grid.best_estimator_

pred = best_model.predict(
    X_test_tfidf
)

print(classification_report(
    y_test,
    pred
))

              precision    recall  f1-score   support

   High Risk       0.31      0.36      0.33        14
      Normal       0.83      0.87      0.85       182
  Suspicious       0.44      0.35      0.39        46

    accuracy                           0.74       242
   macro avg       0.53      0.52      0.52       242
weighted avg       0.73      0.74      0.73       242



In [ ]:
import joblib

joblib.dump(
    best_model,
    "/content/drive/MyDrive/Internship project/best_message_classifier.pkl"
)

joblib.dump(
    vectorizer,
    "/content/drive/MyDrive/Internship project/best_tfidf_vectorizer.pkl"
)

print("Best model saved successfully!")

Best model saved successfully!


In [ ]:
results = pd.DataFrame({
    "Message": X_test.values,
    "Actual_Label": y_test.values,
    "Predicted_Label": pred
})

results.to_csv(
    "/content/drive/MyDrive/Internship project/best_model_predictions.csv",
    index=False
)
